In [4]:
# =============================================================
# PROJECT : NER + Relation Extraction -> Knowledge Graph
# STEP 1: Data loading + subword label alignment
# =============================================================
# WHAT THIS PROJECT DOES (read this first):
#
#   Raw text  ->  [NER model]  ->  entities (PERSON, ORG, LOC)
#             ->  [relation extraction]  ->  (subject, relation, object)
#             ->  [graph builder]  ->  a queryable knowledge graph
#
# Project 1 was SEQUENCE classification (one label per sentence).
# This is TOKEN classification (one label per WORD). That difference
# creates the single hardest concept in this project: label alignment.
#
# THE ALIGNMENT PROBLEM:
#   Your dataset labels WORDS:      ["Elon", "Musk", "founded", "SpaceX"]
#                                   [ B-PER,  I-PER,   O,        B-ORG ]
#   But the tokenizer makes SUBWORDS: ["elon", "mu", "##sk", "founded", "space", "##x"]
#   Now the labels don't line up. We must expand word-labels onto
#   subword-tokens, and mark the "continuation" pieces with -100
#   (PyTorch's ignore_index) so they contribute no loss.
#
# BIO TAGGING SCHEME:
#   B-PER = Beginning of a person entity
#   I-PER = Inside (continuation) of a person entity
#   O     = Outside any entity
#   This is how we encode multi-word entities in a per-token format.
# =============================================================

!pip install datasets transformers seqeval -q

# --- Colab compatibility patch (same torchvision issue as Project 1) ---
import datasets.formatting.torch_formatter as _tf
_tf.config.TORCHVISION_AVAILABLE = False
# -----------------------------------------------------------------------

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

# -------------------------------------------------------------
# 1. Load CoNLL-2003 (the standard NER benchmark)
# -------------------------------------------------------------
# Namespaced path avoids the HfUriError we hit in Project 1.
# NOTE: conll2003 repos (eriktks/conll2003, conllpp, tner/conll2003) all
# ship the old loading-script format, which datasets no longer executes.
# Fix: HF auto-generates a parquet mirror of every dataset (script-based
# or not) on the "refs/convert/parquet" branch — load that directly.
dataset = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")

print(dataset)
print(dataset["train"][0])
# Structure: {'tokens': [...words...], 'ner_tags': [...ints...], ...}

# -------------------------------------------------------------
# 2. Inspect the label scheme
# -------------------------------------------------------------
label_list = dataset["train"].features["ner_tags"].feature.names
print("Labels:", label_list)
# ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}
NUM_LABELS = len(label_list)
print("Num labels:", NUM_LABELS)

# See a real example decoded
ex = dataset["train"][0]
for tok, tag in zip(ex["tokens"], ex["ner_tags"]):
    print(f"{tok:15s} -> {id2label[tag]}")

# -------------------------------------------------------------
# 3. Tokenizer
# -------------------------------------------------------------
MODEL_NAME = "distilbert-base-cased"   # CASED matters for NER!
# Why cased? "Apple" (company) vs "apple" (fruit). Lowercasing
# destroys a huge signal for entity recognition.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -------------------------------------------------------------
# 4. THE CORE FUNCTION: align word-level labels to subword tokens
# -------------------------------------------------------------
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,   # tells HF our input is ALREADY word-split
        max_length=128,
        padding="max_length",
    )

    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        # word_ids() maps each subword token back to its original word index
        # e.g. ["[CLS]","elon","mu","##sk",...] -> [None, 0, 1, 1, ...]
        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                # Special tokens ([CLS], [SEP], padding) -> ignored in loss
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # FIRST subword of a word -> gets the real label
                label_ids.append(word_labels[word_id])
            else:
                # CONTINUATION subword (e.g. "##sk") -> ignored in loss
                label_ids.append(-100)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized

# Apply to all splits
tokenized_ds = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized_ds)

# -------------------------------------------------------------
# 5. VERIFY the alignment worked (do not skip this)
# -------------------------------------------------------------
sample = tokenized_ds["train"][0]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])

print("\n--- Alignment check ---")
for tok, lab in list(zip(tokens, sample["labels"]))[:25]:
    lab_str = "IGNORED" if lab == -100 else id2label[lab]
    print(f"{tok:15s} -> {lab_str}")

# =============================================================
# CHECKPOINT - answer these before moving on:
# 1. Why do continuation subwords get -100 instead of the same label?
# 2. What would break if we used an UNCASED model here?
# 3. In BIO tagging, why do we need both B- and I- prefixes?
#    (Hint: "New York Los Angeles" - how do you know where one
#     location ends and the next begins without B-?)
# =============================================================

conll2003/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

conll2003/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

--- Alignment check ---
[CLS]           -> IGNORED
EU              -> B-ORG
rejects         -> O
German          -> B-MISC
call            -> O
to              -> O
boycott         -> O
British         -> B-MISC
la              -> O
##mb            -> IGNORED
.               -> O
[SEP]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]           -> IGNORED
[PAD]    

In [5]:
# =============================================================
# STEP 2: Train the NER model + evaluate properly
# =============================================================
# CONCEPT: Token classification = a linear classifier on top of
# every token's contextual embedding. Same DistilBERT backbone
# as Project 1, but instead of pooling to ONE vector and
# classifying once, we classify EVERY token position.
#
#   Project 1:  [CLS] w1 w2 w3  ->  pool  ->  1 label
#   Project 3:  [CLS] w1 w2 w3  ->  no pool ->  1 label PER TOKEN
#
# WHY seqeval AND NOT PLAIN ACCURACY:
#   ~85% of NER tokens are "O" (not an entity). A model that
#   predicts "O" for everything scores ~85% token accuracy while
#   being completely useless. seqeval scores ENTITY-LEVEL
#   precision/recall/F1: it only counts an entity correct if the
#   FULL span AND the type both match. That's the honest metric.
# =============================================================

import numpy as np
import torch
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

# -------------------------------------------------------------
# 1. Load the model with a token-classification head
# -------------------------------------------------------------
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,   # baked into the model so inference returns readable tags
    label2id=label2id,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# -------------------------------------------------------------
# 2. Metrics function (entity-level, via seqeval)
# -------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Strip out the -100 positions (special tokens + subword continuations)
    # and convert integer ids back to BIO strings for seqeval.
    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        row_labels, row_preds = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                row_labels.append(id2label[l])
                row_preds.append(id2label[p])
        true_labels.append(row_labels)
        true_preds.append(row_preds)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

# -------------------------------------------------------------
# 3. Train
# -------------------------------------------------------------
data_collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(
    output_dir="./ner_conll2003",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# -------------------------------------------------------------
# 4. Final evaluation on the TEST set
# -------------------------------------------------------------
test_results = trainer.evaluate(tokenized_ds["test"])
print("\n--- Test set results ---")
print(test_results)

# -------------------------------------------------------------
# 5. Per-entity-type breakdown (which types are hardest?)
# -------------------------------------------------------------
predictions, labels, _ = trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions, axis=-1)

true_labels, true_preds = [], []
for pred_row, label_row in zip(preds, labels):
    row_labels, row_preds = [], []
    for p, l in zip(pred_row, label_row):
        if l != -100:
            row_labels.append(id2label[l])
            row_preds.append(id2label[p])
    true_labels.append(row_labels)
    true_preds.append(row_preds)

print("\n--- Per-entity-type report ---")
print(classification_report(true_labels, true_preds))

# Keep these for the README / charts later
ner_test_f1 = test_results["eval_f1"]
ner_test_precision = test_results["eval_precision"]
ner_test_recall = test_results["eval_recall"]

# -------------------------------------------------------------
# 6. Save the model so later steps can load it
# -------------------------------------------------------------
trainer.save_model("./ner_model_final")
tokenizer.save_pretrained("./ner_model_final")
print("\nSaved model to ./ner_model_final")

model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 65,197,833


...

Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.027718,0.104734,3,0.875369,0.893515,0.884349



--- Test set results ---
{'eval_loss': 0.10473434627056122, 'eval_precision': 0.8753688595729908, 'eval_recall': 0.8935152374202693, 'eval_f1': 0.8843489697501097}



--- Per-entity-type report ---
              precision    recall  f1-score   support

         LOC       0.91      0.91      0.91      1666
        MISC       0.71      0.79      0.75       702
         ORG       0.84      0.87      0.86      1661
         PER       0.95      0.95      0.95      1615

   micro avg       0.88      0.89      0.88      5644
   macro avg       0.85      0.88      0.87      5644
weighted avg       0.88      0.89      0.89      5644



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved model to ./ner_model_final


In [6]:
# =============================================================
# STEP 3: Inference — extract entities from raw text
# =============================================================
# CONCEPT: The model outputs a BIO tag per subword token. To get
# usable entities we must:
#   1. Merge subwords back into words
#   2. Merge B-X followed by I-X into ONE entity span
# The `aggregation_strategy="simple"` flag in the HF pipeline
# does step 2 for us, but we also write it manually below so you
# understand what it's actually doing.
# =============================================================

from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
import torch

# -------------------------------------------------------------
# 1. Load your fine-tuned model
# -------------------------------------------------------------
ner_model = AutoModelForTokenClassification.from_pretrained("./ner_model_final")
ner_tokenizer = AutoTokenizer.from_pretrained("./ner_model_final")

device = 0 if torch.cuda.is_available() else -1

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple",   # merges B-X + I-X into single spans
    device=device,
)

# -------------------------------------------------------------
# 2. Test it on real sentences
# -------------------------------------------------------------
test_sentences = [
    "Elon Musk founded SpaceX in California and later acquired Twitter.",
    "Satya Nadella is the CEO of Microsoft, headquartered in Redmond.",
    "Google acquired DeepMind, a research lab based in London.",
    "Imran Khan served as the Prime Minister of Pakistan.",
    "Apple announced a new office in Islamabad last year.",
]

for sent in test_sentences:
    print(f"\nSentence: {sent}")
    entities = ner_pipeline(sent)
    for e in entities:
        print(f"   {e['word']:20s} {e['entity_group']:6s} (conf: {e['score']:.3f})")

# -------------------------------------------------------------
# 3. A clean wrapper we'll reuse in the relation-extraction step
# -------------------------------------------------------------
def extract_entities(text, min_score=0.80):
    """
    Returns a list of dicts:
      [{'text': 'Elon Musk', 'type': 'PER', 'start': 0, 'end': 9}, ...]
    Filters out low-confidence predictions.
    """
    raw = ner_pipeline(text)
    entities = []
    for e in raw:
        if e["score"] < min_score:
            continue
        entities.append({
            "text": e["word"].strip(),
            "type": e["entity_group"],
            "start": e["start"],
            "end": e["end"],
            "score": float(e["score"]),
        })
    return entities

# Sanity check
print("\n--- extract_entities() output ---")
print(extract_entities("Elon Musk founded SpaceX in California."))

# -------------------------------------------------------------
# 4. MANUAL BIO merging (for understanding — not used downstream)
# -------------------------------------------------------------
# This shows what aggregation_strategy="simple" does internally.
def manual_bio_merge(text):
    raw = ner_pipeline.tokenizer(text, return_offsets_mapping=True, truncation=True)
    # Run model without aggregation to see raw per-token tags
    raw_pipe = pipeline("token-classification", model=ner_model,
                        tokenizer=ner_tokenizer, device=device)  # no aggregation
    tagged = raw_pipe(text)

    entities, current = [], None
    for t in tagged:
        tag = t["entity"]
        if tag.startswith("B-"):
            if current:
                entities.append(current)
            current = {"text": t["word"], "type": tag[2:]}
        elif tag.startswith("I-") and current and current["type"] == tag[2:]:
            # continuation — append (handling ## subword pieces)
            piece = t["word"]
            current["text"] += piece[2:] if piece.startswith("##") else " " + piece
        else:
            if current:
                entities.append(current)
            current = None
    if current:
        entities.append(current)
    return entities

print("\n--- manual BIO merge (same result, done by hand) ---")
print(manual_bio_merge("Elon Musk founded SpaceX in California."))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Sentence: Elon Musk founded SpaceX in California and later acquired Twitter.
   Elon Mu              PER    (conf: 0.688)
   ##sk                 ORG    (conf: 0.722)
   SpaceX               ORG    (conf: 0.986)
   California           LOC    (conf: 0.996)
   Twitter              ORG    (conf: 0.977)

Sentence: Satya Nadella is the CEO of Microsoft, headquartered in Redmond.
   Sa                   PER    (conf: 0.995)
   ##tya Nadell         PER    (conf: 0.679)
   Microsoft            ORG    (conf: 0.996)
   Redmond              LOC    (conf: 0.942)

Sentence: Google acquired DeepMind, a research lab based in London.
   Google               ORG    (conf: 0.992)
   DeepMind             ORG    (conf: 0.986)
   London               LOC    (conf: 0.997)

Sentence: Imran Khan served as the Prime Minister of Pakistan.
   Imran Khan           PER    (conf: 0.824)
   Pakistan             LOC    (conf: 0.996)

Sentence: Apple announced a new office in Islamabad last year.
   Apple           